# EX — AI Mastery End-to-End Integration Exercise

Connects ingestion -> retrieval -> generation -> guardrail -> (mock) caching into
one small pipeline, tying together phases 00-09 of this track.


In [ ]:
import re, numpy as np
from collections import Counter

docs = [
    "Refunds take 5-7 business days to process.",
    "Our mobile app supports iOS 15+ and Android 10+.",
    "Support is available 9am-6pm EST for business accounts.",
]

def tokenize(t): return re.findall(r"[a-z]+", t.lower())
vocab = sorted(set(w for d in docs for w in tokenize(d)))
vidx = {w:i for i,w in enumerate(vocab)}
doc_freq = Counter()
for d in docs:
    for w in set(tokenize(d)): doc_freq[w]+=1

def embed(text):
    vec = np.zeros(len(vocab))
    tf = Counter(tokenize(text))
    for w,c in tf.items():
        if w in vidx:
            idf = np.log((1+len(docs))/(1+doc_freq[w]))+1
            vec[vidx[w]] = c*idf
    return vec

doc_vecs = np.array([embed(d) for d in docs])

def cosine(a,b):
    denom = np.linalg.norm(a)*np.linalg.norm(b)
    return 0.0 if denom==0 else np.dot(a,b)/denom


### TODO 1
Build `pipeline(query)` that: (1) checks an in-memory `cache` dict first, (2) retrieves top-1 doc, (3) if best similarity < 0.05 returns "I don't know", (4) otherwise returns a mock grounded answer, (5) stores the result in `cache` before returning.

In [ ]:
cache = {}

# TODO
def pipeline(query):
    pass

print(pipeline("how long do refunds take"))
print(pipeline("how long do refunds take"))  # should hit cache second time
print("cache size:", len(cache))


<details><summary>Solution</summary>

```python
def pipeline(query):
    if query in cache:
        return cache[query] + " [from cache]"
    qvec = embed(query)
    sims = [cosine(qvec, dv) for dv in doc_vecs]
    best_idx = int(np.argmax(sims))
    if sims[best_idx] < 0.05:
        answer = "I don't know based on the available information."
    else:
        answer = f"[grounded in]: {docs[best_idx]}"
    cache[query] = answer
    return answer
```
</details>


## Key Takeaways
- A production pipeline is retrieval + generation + guardrail (threshold check) + caching, composed together.
- Caching identical queries is a simple, high-leverage production optimization.
- A similarity threshold is a cheap way to avoid answering when there's no good retrieved match.
